# UCS420 Lab Assignment 7 — NLP Text Preprocessing

Aadish Jain

Roll No: 1024170413

Batch: 3Q33

In [ ]:
ROLL_NO = 1024170413  # Student roll number
NAME = "Aadish Jain"

import nltk, re, math, string
from collections import Counter
import matplotlib.pyplot as plt
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [ ]:
# Q1: Run the official pipeline for every generated ticket.
processed = [process_ticket(text) for text in df["Ticket_Text"]]
q1_rows = []
for ticket_id, result in zip(df["Ticket_ID"], processed):
    q1_rows.append({
        "Ticket ID": ticket_id,
        "Number of sentences": len(result["sentences"]),
        "Number of tokens": len(result["tokens"]),
        "Punctuation tokens": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique tokens": len(set(result["tokens"])),
        "Final tokens": len(result["final"])
    })
q1_table = pd.DataFrame(q1_rows)
total = {"Ticket ID": "TOTAL"}
for col in ["Number of sentences", "Number of tokens", "Punctuation tokens", "Stopwords", "Final tokens"]:
    total[col] = int(q1_table[col].sum())
total["Unique tokens"] = len(set(t for result in processed for t in result["tokens"]))
display(pd.concat([q1_table, pd.DataFrame([total])], ignore_index=True))

# Display actual token lists for one focus ticket, using the generated focus IDs.
focus_id = FOCUS[0]
focus_index = df.index[df["Ticket_ID"] == focus_id][0]
focus_result = processed[focus_index]
print(f"Selected focus ticket: {focus_id}")
print("Raw ticket:", df.loc[focus_index, "Ticket_Text"])
print("Actual tokens:", focus_result["tokens"])
print("Punctuation tokens:", focus_result["punct"])
print("Stopwords:", focus_result["stops"])
print("Final tokens:", focus_result["final"])

# Corpus lists reused by Q3-Q5.
S1 = [t for result in processed for t in result["tokens"]]
S2 = [t for result in processed for t in result["tokens"] if not is_punct(t)]
S3 = [t for result in processed for t in result["final"]]


**Your answer:** Tokens such as `n't`, `'s`, and `ca` can appear because NLTK's `word_tokenize()` splits contractions into separate pieces (for example, `can't` can become `ca` + `n't`). The official pipeline lowercases tokens and removes stopwords and punctuation, but it does not recombine contraction fragments; therefore some fragments may remain in the final-token list.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]


# Q2: Extract patterns from raw tickets and the required TEST_LINES.
import re
import re
regex_patterns = {
    "Words longer than 5 letters": r"\b[A-Za-z]{6,}\b",
    # Match dotted versions/decimals first (2.0.1, 95.5), then integers/hex-like codes.
    "Numbers (versions/decimals kept whole)": r"(?<![\w.])(?:0[xX][0-9A-Fa-f]+|\d+(?:\.\d+)+|\d[\d,]*)(?![\w.])",
    "Capitalized words": r"\b[A-Z][a-zA-Z]*\b",
    "Words beginning with a vowel": r"\b[AEIOUaeiou][A-Za-z]*\b"
}
raw_tickets = df["Ticket_Text"].tolist()
for label, pattern in regex_patterns.items():
    print("\\n" + label + " — pattern:", pattern)
    print("TEST_LINES:")
    for line in TEST_LINES: print(re.findall(pattern, line))
    print("TICKETS:")
    for ticket_id, line in zip(df["Ticket_ID"], raw_tickets):
        print(ticket_id, re.findall(pattern, line))

email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
url_pattern = r"\b(?:https?://|www\.)[^\s<>]+"
phone_pattern = r"(?<!\w)(?:\+\d{1,3}[ -]?)?(?:\(?\d{3,5}\)?[ -]?)\d{3,5}[ -]?\d{3,5}(?!\w)"
def redact_pii(text):
    # URL first so digits in URLs do not get mistaken for phone numbers.
    text = re.sub(url_pattern, lambda m: "<URL>" + ("" if not m.group(0) or m.group(0)[-1] not in ".,;!?" else m.group(0)[-1]), text, flags=re.IGNORECASE)
    text = re.sub(email_pattern, "<EMAIL>", text)
    return re.sub(phone_pattern, "<PHONE>", text)

print("\\nReplacement checks on TEST_LINES:")
for line in TEST_LINES: print("IN :", line, "\\nOUT:", redact_pii(line))
print("\\nReplacement checks on generated tickets:")
for ticket_id, line in zip(df["Ticket_ID"], raw_tickets):
    print(ticket_id, "=>", redact_pii(line))


**Your answer:** The exact count is computed below from the generated tickets. Some capitalized matches are sentence-initial words rather than proper names/acronyms; the regex only detects capitalization and cannot determine why a word is capitalized.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
# Q3: A single RegexpTokenizer pattern, with specific forms listed before generic words.
from nltk.tokenize import RegexpTokenizer

TOKEN_PATTERN = r"""(?ix)
    (?:https?://|www\.)[^\s<>]+
  | [A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}
  | \+?\d{1,3}[ -]\d{5}[ -]\d{5}
  | \b\d{4,5}-\d{3,5}-\d{3,5}\b
  | \b\d+(?:\.\d+)+\b
  | \b\d+\.\d+\b
  | \b[\w]+(?:['’][\w]+)+\b
  | \b[\w]+(?:-[\w]+)+\b
  | \b\w+\b
  | [^\w\s]
"""
custom_tokenizer = RegexpTokenizer(TOKEN_PATTERN)
def my_tokenize(text):
    return custom_tokenizer.tokenize(text)

print("TEST_LINES comparison")
for line in TEST_LINES:
    print("\\nTEXT:", line)
    print("my_tokenize :", my_tokenize(line))
    print("word_tokenize:", word_tokenize(line))

ticket_token_differences = []
for ticket_id, line in zip(df["Ticket_ID"], raw_tickets):
    mine, standard = my_tokenize(line), word_tokenize(line)
    if mine != standard:
        ticket_token_differences.append(ticket_id)
    print(f"\\n{ticket_id}: custom={mine}\\n       nltk={standard}")
print("\\nTickets tokenized differently:", len(ticket_token_differences), "/", len(raw_tickets), ticket_token_differences)

# Concrete differences to cite in the written interpretation.
for line in TEST_LINES:
    if my_tokenize(line) != word_tokenize(line):
        print("\\nExample difference:")
        print("Text:", line)
        print("Custom:", my_tokenize(line))
        print("word_tokenize:", word_tokenize(line))
        break


**Your answer:** The number of generated tickets whose token lists differ is printed by the preceding cell. Differences are expected for forms the custom regex intentionally preserves as one token, such as email addresses, URLs, phone numbers, decimals/versions, contractions, and hyphenated words; NLTK's general tokenizer may split those forms.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
# Q4: Compare stemmers and POS-aware WordNet lemmatization on unique final tokens.
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
porter = PorterStemmer()
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()
unique_final = sorted(set(S3))

# Choose a suffix based on actual corpus evidence; prefer the most frequent supported suffix.
suffix_candidates = ["ing", "ed", "ies", "es", "s", "ly", "ment"]
suffix_evidence = {
    suffix: [w for w in unique_final if len(w) > len(suffix) + 2 and w.endswith(suffix)]
    for suffix in suffix_candidates
}
supported = [(suffix, words) for suffix, words in suffix_evidence.items() if len(words) >= 2]
if supported:
    chosen_suffix, supporting_words = max(supported, key=lambda item: len(item[1]))
else:
    chosen_suffix, supporting_words = max(suffix_evidence.items(), key=lambda item: len(item[1]))
regexp_stemmer = RegexpStemmer(chosen_suffix + "$", min=4)
print("RegexpStemmer suffix selected:", "-" + chosen_suffix)
print("Supporting words in this dataset:", supporting_words)
print("Justification: this suffix was selected after inspecting the generated corpus; the listed words provide dataset-specific evidence. A regexp stemmer strips the suffix mechanically and can over-stem.")

def penn_to_wordnet(tag):
    if tag.startswith("J"): return "a"
    if tag.startswith("V"): return "v"
    if tag.startswith("N"): return "n"
    if tag.startswith("R"): return "r"
    return "n"

# Assign the first observed POS tag for each final token from the original ticket context.
pos_by_word = {}
for text in raw_tickets:
    for word, tag in nltk.pos_tag(word_tokenize(text)):
        key = word.lower()
        if key in unique_final and key not in pos_by_word:
            pos_by_word[key] = penn_to_wordnet(tag)

stem_rows = []
for word in unique_final:
    pos = pos_by_word.get(word, "n")
    row = {
        "Word": word,
        "Porter": porter.stem(word),
        "Lancaster": lancaster.stem(word),
        "Regexp": regexp_stemmer.stem(word),
        "WordNet lemma (POS-aware)": lemmatizer.lemmatize(word, pos=pos),
        "POS": pos
    }
    if len({row[k] for k in ["Porter", "Lancaster", "Regexp", "WordNet lemma (POS-aware)"]}) > 1:
        stem_rows.append(row)
stem_df = pd.DataFrame(stem_rows, columns=["Word","Porter","Lancaster","Regexp","WordNet lemma (POS-aware)","POS"])
print("Unique final tokens:", len(unique_final), "| Words where outputs differ:", len(stem_df))
display(stem_df)


**Your answer:** Use the examples printed below to identify one over-stemming case (a stem becomes too short or loses the word's recognizable meaning) and one under-stemming case (related forms are not reduced to the same stem). Stemming is rule-based and does not consult word meaning; compare the Porter/Lancaster outputs with the WordNet lemma in the table.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
# Q5: Corpus statistics, frequent words, Zipf-style plot, and bigrams.
import math
from collections import Counter
import matplotlib.pyplot as plt
S4 = [porter.stem(token) for token in S3]

def corpus_statistics(name, tokens):
    total = len(tokens)
    unique = len(set(tokens))
    return {"Corpus": name, "Total tokens": total, "Unique tokens": unique,
            "TTR": unique / total if total else 0.0}

statistics = pd.DataFrame([
    corpus_statistics("S1: all tokens", S1),
    corpus_statistics("S2: without punctuation", S2),
    corpus_statistics("S3: final tokens", S3),
    corpus_statistics("S4: S3 after Porter stemming", S4)
])
display(statistics.style.format({"TTR": "{:.4f}"}))

def show_top_words(tokens, n=10):
    return pd.DataFrame(Counter(tokens).most_common(n), columns=["Word","Frequency"])
print("Top 10 words — S1"); display(show_top_words(S1, 10))
print("Top 10 words — S3"); display(show_top_words(S3, 10))

frequencies = sorted(Counter(S3).values(), reverse=True)
ranks = range(1, len(frequencies) + 1)
plt.figure(figsize=(8,5))
plt.plot([math.log(r) for r in ranks], [math.log(f) for f in frequencies], marker="o", linewidth=1)
plt.xlabel("log(rank)"); plt.ylabel("log(frequency)")
plt.title("S3: log(rank) vs log(frequency)")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

def show_top_bigrams(tokens, n=5):
    pairs = list(zip(tokens[:-1], tokens[1:]))
    return pd.DataFrame(Counter(pairs).most_common(n), columns=["Bigram","Frequency"])
print("Top 5 bigrams — S1"); display(show_top_bigrams(S1, 5))
print("Top 5 bigrams — S3"); display(show_top_bigrams(S3, 5))


**Your answer:** TTR is the number of unique token types divided by total tokens. From S1 to S4, punctuation and stopwords are removed to form S3, then Porter stemming maps some related surface forms to the same stem. The exact TTR movement depends on this roll-number-specific dataset; use the printed table to report whether it rises or falls and by how much.

---
## Q6. Interpretation
Answer the four questions using the outputs generated above. The code below prints examples from this specific dataset where possible.

In [ ]:
# (a) TTR interpretation from this dataset.
print("(a) TTR changes because S1 includes all tokens, S3 removes punctuation and stopwords, and S4 stems S3. Stemming can merge multiple word forms into one type; the exact values are in the corpus-statistics table.")
# (b) One concrete tokenizer difference.
difference = next(((text, my_tokenize(text), word_tokenize(text)) for text in list(TEST_LINES)+raw_tickets if my_tokenize(text) != word_tokenize(text)), None)
if difference:
    print("\n(b) Example:", difference[0])
    print("my_tokenize:", difference[1])
    print("word_tokenize:", difference[2])
    print("Explanation: the regex tokenizer preserves selected formats such as hyphenated words, contractions, versions, emails, URLs, and phone numbers; NLTK may split them.")
else:
    print("\n(b) No difference found; inspect tokenizer pattern and test lines.")
# (c) An example of potentially undesirable stemming.
over = next(((w, porter.stem(w)) for w in unique_final if porter.stem(w) != w and (len(porter.stem(w)) <= 3 or porter.stem(w) not in unique_final)), None)
if over:
    print(f"\n(c) Potentially undesirable stemming: {over[0]} -> {over[1]}. A stem may not be a standalone dictionary word.")
else:
    print("\n(c) Inspect the stem comparison table for a stem that is not a dictionary word; mechanical suffix removal may discard meaningful information.")
# (d) POS-aware lemma example where it differs from Porter.
lemma_example = next(((w, pos_by_word.get(w,"n"), lemmatizer.lemmatize(w,pos=pos_by_word.get(w,"n")), porter.stem(w))
                      for w in unique_final
                      if lemmatizer.lemmatize(w,pos=pos_by_word.get(w,"n")) != porter.stem(w)
                      and lemmatizer.lemmatize(w,pos=pos_by_word.get(w,"n")) != w), None)
if lemma_example:
    print(f"\n(d) POS-aware lemmatization: {lemma_example[0]} (POS={lemma_example[1]}) -> {lemma_example[2]}, while Porter gives {lemma_example[3]}. WordNet uses lexical knowledge and the grammatical category to return a dictionary lemma.")
else:
    print("\n(d) See the comparison table for a word where WordNet's POS-aware lemma is a dictionary form while the stemmer output is only a stem.")
